## Example Notebook: Using IngestionPipeline

### 🛠️ Setup Instructions

Before running this notebook:
- Make sure all required libraries are installed by running:
```bash
    pip install ".[video-agent]"
```


---

### About

The **IngestionPipeline** performs comprehensive processing of video file to extract transcript, keyframes, chapters, object registry, ai search index creation for downstream applications - `VideoAgent`. It includes the following steps:

1. **Audio Extraction** – Extracts the audio from the input video.
2. **Transcription** – Converts spoken content to text using the provided transcription provider and language setting.
3. **KeyFrame Extraction** – Extract Keyframes using the calcOpticalFlowFarneback algorithm.
4. **Chapter Generation** – Aligns transcript segments with visual frames to form meaningful video chapters.
5. **Object Collection Generation** - Generate object collection for a video for object tracking.
5. **Vector Search Indexing** – Saves chapters, object collection, keyframe and related metadata to an Vector Search index to support retrieval for different needs.
---

### Transcription Configuration
- Provide Valid Transcripton provider
Specify the language of the video's audio using the `Languages` enum. For example:

- `Languages.ENGLISH_INDIA` – English (India)
- `Languages.HINDI` – Hindi

The `Languages` enum includes support for additional languages. Refer to the `Languages` enum definition to explore all available options.


### Importing Libaries

In [ ]:
from mmct.video_pipeline import IngestionPipeline, Languages
from mmct.config.providers import IngestionProviderConfig
from mmct.providers.azure import (
    AzureLLMProvider,
    AzureEmbeddingProvider,
    AzureSearchProvider,
    AzureStorageProvider,
    WhisperTranscriptionProvider,
)
from mmct.providers.local import CustomImageEmbeddingProvider
from azure.identity import DefaultAzureCredential, AzureCliCredential, ChainedTokenCredential
import nest_asyncio

nest_asyncio.apply()

### Configure the IngestionProviderConfig

In [ ]:
credentials = ChainedTokenCredential(AzureCliCredential(), DefaultAzureCredential())

In [ ]:
provider = IngestionProviderConfig(
    llm_provider=AzureLLMProvider(
        endpoint="https://<your-openai-endpoint>.openai.azure.com/",
        deployment_name="<your-llm-deployment-name>",
        model_name="<your-llm-model-name>",
        api_version="<your-api-version>",
        credentials=credentials,
    ),
    embedding_provider=AzureEmbeddingProvider(
        endpoint="https://<your-openai-endpoint>.openai.azure.com/",
        deployment_name="<your-embedding-deployment-name>",
        api_version="<your-api-version>",
        credentials=credentials,
    ),
    image_embedding_provider=CustomImageEmbeddingProvider(),
    vectordb_chapter=AzureSearchProvider(
        endpoint="https://<your-search-service>.search.windows.net",
        index_name="<your-chapter-index-name>",
        credentials=credentials,
    ),
    vectordb_keyframes=AzureSearchProvider(
        endpoint="https://<your-search-service>.search.windows.net",
        index_name="<your-keyframe-index-name>",
        credentials=credentials,
    ),
    vectordb_object_registry=AzureSearchProvider(
        endpoint="https://<your-search-service>.search.windows.net",
        index_name="<your-object-registry-index-name>",
        credentials=credentials,
    ),
    storage_provider=AzureStorageProvider(
        storage_account_name="<your-storage-account-name>",
        keyframe_container_name="<your-keyframe-container-name>",
        credentials=credentials,
    ),
    transcription_provider=WhisperTranscriptionProvider(
        endpoint="https://<your-openai-endpoint>.openai.azure.com/",
        api_version="<your-api-version>",
        deployment_name="<your-whisper-deployment-name>",
        credentials=credentials,
    ),
)

* You can use api_key/blob_connection_string instead of credentials for these providers.

### Executing Video Pipeline

In [ ]:
keyframe_config = {"motion_threshold": 1.5, "sample_fps": 2}  # Example keyframe extraction config
url = "video-url"
video_path = "local-path-to-video-file"
transcript_path = (
    None  # "local-path-to-transcript-file" #External transcript file path if available
)
source_language = Languages.ENGLISH_UNITED_STATES


# Create IngestionPipeline instance
ingestion = IngestionPipeline(
    video_path=video_path,
    language=source_language,
    transcript_path=transcript_path,  # Optional: provide if external transcript file is available
    keyframe_config=keyframe_config,
    url=url,  # Optional: provide if video is from a URL
    provider=provider,
)

# Run the ingestion pipeline
await ingestion.run()